# RT-DETR-X 2-class continue training

Notebook nay dung de train tiep tu `best.pt` da co san.

- Dau vao: best model cua run truoc + runtime dataset 2 class.
- Tu tinh `EXTRA_EPOCHS` dua tren `EXTRA_HOURS_TARGET` va toc do epoch uoc tinh.
- Phu hop khi muon mo rong them sau notebook train chinh.


In [ ]:
!pip -q install -U ultralytics pyyaml

import csv
import gc
import json
import time
import zipfile
from pathlib import Path

import torch
import yaml
from ultralytics import RTDETR

print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), '| GPU count:', torch.cuda.device_count())


In [ ]:
# ===== User config =====
# Dat vao path best.pt cua run truoc (trong /kaggle/input hoac /kaggle/working)
BASE_BEST_PATH = '/kaggle/input/your-prev-bundle/weights/best.pt'

# Dat vao runtime yaml 2-class (neu khong dat, notebook se tim trong /kaggle/input va /kaggle/working)
RUNTIME_YAML_HINT = '/kaggle/input/your-prev-bundle/data_runtime_2class_rtdetr.yaml'

RUN_NAME = 'rtdetr_x_2class_continue_v1'
PROJECT_DIR = Path('/kaggle/working/runs/detect')
REPORT_JSON = Path('/kaggle/working/report_metrics_rtdetr_2class_continue.json')
REPORT_CSV = Path('/kaggle/working/report_metrics_rtdetr_2class_continue.csv')

IMAGE_SIZE = 1024
EXTRA_HOURS_TARGET = 3.0
DEFAULT_MIN_PER_EPOCH = 17.0   # fallback neu khong tinh duoc tu results.csv
MIN_EXTRA_EPOCHS = 6
MAX_EXTRA_EPOCHS = 40

TRAIN_DEVICE = '0,1' if torch.cuda.is_available() and torch.cuda.device_count() >= 2 else ('0' if torch.cuda.is_available() else 'cpu')
EVAL_DEVICE = 0 if torch.cuda.is_available() else 'cpu'
BATCH_CANDIDATES = [4, 2, 1] if torch.cuda.device_count() >= 2 else ([2, 1] if torch.cuda.is_available() else [1])
EVAL_BATCH = 1

print('Train device     :', TRAIN_DEVICE)
print('Batch candidates :', BATCH_CANDIDATES)


In [ ]:
def read_yaml(path):
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

def find_runtime_yaml(hint):
    if hint:
        p = Path(hint)
        if p.exists():
            return p.resolve()
    cands = []
    for root in [Path('/kaggle/input'), Path('/kaggle/working')]:
        if not root.exists():
            continue
        for p in root.rglob('*.yaml'):
            s = p.as_posix().lower()
            if 'runtime' in s and '2class' in s:
                cands.append(p.resolve())
    if not cands:
        raise FileNotFoundError('Khong tim thay runtime yaml 2class. Hay dat RUNTIME_YAML_HINT')
    cands.sort(key=lambda x: len(str(x)))
    return cands[0]

def estimate_min_per_epoch(base_best):
    run_dir = base_best.parent.parent
    results_csv = run_dir / 'results.csv'
    if not results_csv.exists():
        return None
    try:
        rows = [r for r in csv.DictReader(results_csv.open('r', encoding='utf-8'))]
    except Exception:
        return None
    if len(rows) < 2:
        return None
    # Ultralytics results.csv khong luu thoi gian tung epoch, dung estimate co dinh neu khong co thong tin tot hon
    return None

base_best = Path(BASE_BEST_PATH)
if not base_best.exists():
    raise SystemExit(f'Khong tim thay BASE_BEST_PATH: {base_best}')

runtime_yaml = find_runtime_yaml(RUNTIME_YAML_HINT)
runtime_cfg = read_yaml(runtime_yaml)

if not isinstance(runtime_cfg, dict) or ('train' not in runtime_cfg) or ('val' not in runtime_cfg):
    raise SystemExit('Runtime yaml khong hop le cho detect train/val')

min_per_epoch = estimate_min_per_epoch(base_best)
if min_per_epoch is None:
    min_per_epoch = float(DEFAULT_MIN_PER_EPOCH)

extra_epochs = int(round((float(EXTRA_HOURS_TARGET) * 60.0) / max(1e-6, float(min_per_epoch))))
extra_epochs = max(int(MIN_EXTRA_EPOCHS), min(int(MAX_EXTRA_EPOCHS), extra_epochs))

print('Base best   :', base_best)
print('Runtime yaml:', runtime_yaml)
print('Est min/ep  :', round(min_per_epoch, 3))
print('Extra epochs:', extra_epochs)


In [ ]:
def empty_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def train_with_batch_fallback(model_source, run_name, train_args, batch_candidates):
    last_error = None
    for b in batch_candidates:
        empty_cache()
        t0 = time.time()
        print(f'\\nStart {run_name} | model={model_source} | batch={b}')
        try:
            model = RTDETR(str(model_source))
            model.train(name=run_name, batch=int(b), **train_args)
            elapsed_min = (time.time() - t0) / 60.0
            save_dir = Path(train_args['project']) / run_name
            return {
                'ok': True,
                'batch': int(b),
                'elapsed_min': float(elapsed_min),
                'save_dir': str(save_dir),
                'error': None,
            }
        except Exception as e:
            last_error = e
            msg = str(e).lower()
            print(f'[train] batch={b} failed: {e}')
            if ('out of memory' in msg) or ('cuda error' in msg) or ('cudnn' in msg):
                continue
            raise
    raise RuntimeError(f'All batch attempts failed. Last error: {last_error}')

continue_args = dict(
    data=str(runtime_yaml),
    device=TRAIN_DEVICE,
    epochs=int(extra_epochs),
    imgsz=IMAGE_SIZE,
    optimizer='AdamW',
    lr0=0.00008,
    lrf=0.05,
    weight_decay=0.0005,
    warmup_epochs=2.0,
    cos_lr=True,
    pretrained=True,
    amp=True,
    cache=False,
    workers=4,
    save=True,
    save_period=0,
    plots=True,
    val=True,
    verbose=True,
    project=str(PROJECT_DIR),
    exist_ok=True,
    seed=42,
    deterministic=False,
    patience=max(6, int(round(extra_epochs * 0.5))),
    hsv_h=0.010,
    hsv_s=0.35,
    hsv_v=0.2,
    degrees=0.5,
    translate=0.02,
    scale=0.12,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.0,
    mixup=0.0,
    close_mosaic=1,
    erasing=0.0,
)

train_info = train_with_batch_fallback(base_best, RUN_NAME, continue_args, BATCH_CANDIDATES)
run_dir = Path(train_info['save_dir'])
best_path = run_dir / 'weights' / 'best.pt'
last_path = run_dir / 'weights' / 'last.pt'
assert best_path.exists(), f'Missing best after continue: {best_path}'

print('train_info:', train_info)
print('best_path :', best_path)


In [ ]:
def pack_metrics(metrics):
    return {
        'mAP50': float(metrics.box.map50),
        'mAP50-95': float(metrics.box.map),
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
    }

empty_cache()
model = RTDETR(str(best_path))
val_m = model.val(data=str(runtime_yaml), split='val', device=EVAL_DEVICE, imgsz=IMAGE_SIZE, batch=EVAL_BATCH, verbose=True)

test_pack = None
runtime_cfg = read_yaml(runtime_yaml)
if 'test' in runtime_cfg:
    test_m = model.val(data=str(runtime_yaml), split='test', device=EVAL_DEVICE, imgsz=IMAGE_SIZE, batch=EVAL_BATCH, verbose=True)
    test_pack = pack_metrics(test_m)

report = {
    'base_best': str(base_best),
    'runtime_yaml': str(runtime_yaml),
    'run_name': RUN_NAME,
    'image_size': IMAGE_SIZE,
    'extra_hours_target': EXTRA_HOURS_TARGET,
    'estimated_min_per_epoch': min_per_epoch,
    'extra_epochs': extra_epochs,
    'train_info': train_info,
    'best_path': str(best_path),
    'val': pack_metrics(val_m),
    'test': test_pack,
}

REPORT_JSON.write_text(json.dumps(report, indent=2), encoding='utf-8')
with REPORT_CSV.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['split', 'mAP50', 'mAP50-95', 'precision', 'recall'])
    writer.writeheader()
    writer.writerow({'split': 'val', **report['val']})
    if report['test']:
        writer.writerow({'split': 'test', **report['test']})

print(json.dumps(report, indent=2))


In [ ]:
bundle = Path('/kaggle/working') / f'{RUN_NAME}_bundle.zip'
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in [runtime_yaml, REPORT_JSON, REPORT_CSV, best_path, last_path]:
        if p.exists():
            arc = f'weights/{p.name}' if p.suffix == '.pt' else p.name
            zf.write(p, arcname=arc)
    for rel in ['args.yaml', 'results.csv', 'results.png', 'PR_curve.png', 'P_curve.png', 'R_curve.png', 'F1_curve.png']:
        p = run_dir / rel
        if p.exists():
            zf.write(p, arcname=f'continue/{rel}')

print('Bundle saved:', bundle)
print('Size MB:', round(bundle.stat().st_size / (1024 * 1024), 2))
